# Tutorial t-SNE en Python: buenas prácticas

**Jesús F García Gavilán**

2026-06-21

Versión equivalente en Python del tutorial en R (`Rtsne` + `ggplot2`).

## 1. Librerías

`scikit-learn` implementa t-SNE de forma eficiente (con aproximación Barnes-Hut, igual que `Rtsne`). Usamos `pandas`/`numpy` para los datos y `seaborn`/`matplotlib` para la visualización.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_theme(style="whitegrid")

## 2. Reproducibilidad

La semilla se fija ANTES de cualquier paso estocástico (t-SNE inicializa las posiciones de forma aleatoria, así que esto es imprescindible).

In [ ]:
np.random.seed(123)

## 3. Datos

Simulamos un dataset con estructura de clústeres conocida, para poder comprobar después si t-SNE recupera esa estructura. En un caso real, esto sería vuestra matriz de muestras x variables (genes, metabolitos, biomarcadores...).

In [ ]:
n_por_grupo = 100
n_variables = 50

grupo_a = np.random.normal(loc=0, scale=1, size=(n_por_grupo, n_variables))
grupo_b = np.random.normal(loc=3, scale=1, size=(n_por_grupo, n_variables))
grupo_c = np.random.normal(loc=-3, scale=2, size=(n_por_grupo, n_variables))

datos = np.vstack([grupo_a, grupo_b, grupo_c])
etiquetas = np.repeat(["Grupo A", "Grupo B", "Grupo C"], n_por_grupo)

## 4. Preprocesado

### 4.1 Eliminar duplicados exactos

Al igual que `Rtsne`, conviene comprobar y eliminar duplicados antes de seguir.

In [ ]:
df_datos = pd.DataFrame(datos)
duplicados = df_datos.duplicated()

if duplicados.any():
    datos = datos[~duplicados.values]
    etiquetas = etiquetas[~duplicados.values]

### 4.2 Escalar los datos ANTES de pasarlos a TSNE, como paso independiente

Esto evita confundir "varianza alta porque la variable está en otra escala" con "varianza alta porque hay señal real".

In [ ]:
escalador = StandardScaler()
datos_escalados = escalador.fit_transform(datos)

## 5. Reducción inicial con PCA

A diferencia de `Rtsne` (que integra esta reducción con el argumento `pca = TRUE`), `scikit-learn` no la incluye dentro de `TSNE`: hay que hacerla como paso explícito, tal y como recomienda el paper original (reducir a unas 30 dimensiones antes de calcular las afinidades).

In [ ]:
pca = PCA(n_components=30, random_state=123)
datos_pca = pca.fit_transform(datos_escalados)

## 6. Elegir la perplejidad

Regla práctica: la perplejidad debe ser mucho menor que el número de muestras (como mucho un tercio, normalmente bastante menos). Con n = 300 muestras, probamos varios valores para ver el efecto.

In [ ]:
perplejidades = [5, 30, 50]
resultados = {}

for p in perplejidades:
    tsne = TSNE(
        n_components=2,
        perplexity=p,
        method="barnes_hut",   # equivalente al theta de Rtsne
        angle=0.5,             # mismo compromiso precisión/velocidad que theta = 0.5
        init="random",         # ya hemos reducido con PCA a mano, partimos de init aleatorio
        random_state=123,
        n_iter=1000,           # en sklearn >= 1.5 este argumento se llama max_iter
        verbose=1,
    )
    resultados[p] = tsne.fit_transform(datos_pca)

Comentarios para el ajuste:

- `angle` → equivalente al `theta` de Rtsne (aproximación de Barnes-Hut); 0 = exacto (lento), 0.5 = buen compromiso.
- La reducción con PCA se hace ANTES, a mano (sección 5): aquí no existe el argumento `pca` que tiene `Rtsne`.
- `init="random"`: si no hubiéramos reducido antes con PCA, también podríamos usar `init="pca"` para que scikit-learn inicialice el embedding con esa proyección, pero ya la hemos hecho nosotros de forma explícita.

## 7. Construir DataFrames para graficar

In [ ]:
def construir_df(resultado, etiquetas, perplejidad):
    return pd.DataFrame({
        "Dim1": resultado[:, 0],
        "Dim2": resultado[:, 1],
        "Grupo": etiquetas,
        "Perplejidad": f"Perplejidad = {perplejidad}",
    })

df_completo = pd.concat(
    [construir_df(resultados[p], etiquetas, p) for p in perplejidades],
    ignore_index=True,
)

## 8. Visualización

In [ ]:
g = sns.relplot(
    data=df_completo,
    x="Dim1", y="Dim2",
    hue="Grupo",
    col="Perplejidad",
    kind="scatter",
    s=40, alpha=0.7,
    facet_kws={"sharex": False, "sharey": False},
    height=4.5, aspect=1,
)

g.fig.suptitle(
    "t-SNE: efecto de la perplejidad sobre la separación de clústeres",
    y=1.05,
)
g.set_titles("{col_name}")
g.set_axis_labels("Dimensión 1", "Dimensión 2")

plt.show()

## 9. Notas de buenas prácticas (a modo de chuleta)

- Las distancias ENTRE clústeres en el plot no significan nada; solo importa qué cae dentro de cada uno.
- Si necesitáis reproducibilidad exacta, documentad la semilla Y la versión de `scikit-learn` (la implementación de `TSNE` ha cambiado entre versiones).
- Probad siempre más de un valor de perplejidad antes de quedaros con una conclusión visual; un único gráfico puede engañar.
- t-SNE es para explorar, no para inferir. No uséis las coordenadas resultantes como input de un modelo predictivo "porque ya están en 2D".
- A diferencia de R, aquí la reducción con PCA no viene integrada: hacedla siempre como paso explícito antes de `TSNE` (sección 5).